# VCOD development visualizations

Development-only views for dataset samples and exploratory runs. These plots are not part of the locked final report produced by notebook 07.

## Paired COD sample

In [ ]:
from cod_ssl.data import CODDataset
from cod_ssl.data.transforms import MEAN, STD
import matplotlib.pyplot as plt
import torch
dataset = CODDataset('manifests/train_dev.csv', training=False)
sample = dataset[0]
image = sample['image'] * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image.permute(1, 2, 0).clamp(0, 1)); axes[0].set_title(sample['id'])
axes[1].imshow(sample['mask'][0], cmap='gray'); axes[1].set_title('binary mask')
for axis in axes: axis.axis('off')
plt.tight_layout()

## Exploratory 2×2 dashboard

Point `EXPLORATORY_ROOT` at the Drive exploratory directory or an extracted download. Only completed, video-balanced runs are included.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

EXPLORATORY_ROOT = Path('/content/drive/MyDrive/cod-ssl/vcod/exploratory')
# Local example:
# EXPLORATORY_ROOT = Path('/Users/panagiotis/Downloads/drive-download-20260903T065219Z-1-001')

records, summaries = [], {}
for summary_path in sorted(EXPLORATORY_ROOT.rglob('summary.json')):
    summary = json.loads(summary_path.read_text())
    run, limits = summary['run'], summary.get('exploratory_limits', {})
    system = run.get('system_id')
    if system not in {'DS', 'VI', 'DT', 'VV'}:
        continue
    if limits.get('sampling') != 'deterministic_video_balanced':
        continue
    if not (summary_path.parent / 'EVALUATION_COMPLETE').is_file():
        continue
    if system in summaries:
        raise ValueError(f'Duplicate completed balanced exploratory run for {system}')
    summaries[system] = (summary_path, summary)
    metrics = summary['metrics']['minmax']['video_weighted_study_primary']
    records.append({'system': system, **metrics,
                    'ms_per_frame': summary['timing']['ms_per_output_frame'],
                    'peak_memory_mb': summary['timing']['peak_gpu_memory_mb']})

system_order = ['DS', 'VI', 'DT', 'VV']
missing = set(system_order) - set(summaries)
if missing:
    raise ValueError(f'Incomplete balanced exploratory 2×2; missing {sorted(missing)}')
results = pd.DataFrame(records).set_index('system').loc[system_order]
display(results.round(4))

In [ ]:
# Headline metrics. MAE is inverted so taller bars consistently mean better.
plot_values = results[['S', 'weightedF', 'E_adapt', 'E_mean']].copy()
plot_values['1 − MAE'] = 1 - results['MAE']
ax = plot_values.plot.bar(figsize=(12, 5), width=0.78)
ax.set(title='Balanced exploratory validation — video-weighted metrics',
       xlabel='System', ylabel='Score (higher is better)', ylim=(0, 1))
ax.legend(ncol=5, loc='lower center', bbox_to_anchor=(0.5, 1.01), frameon=False)
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()

In [ ]:
# Quality/compute trade-offs; the preferred region is upper-left.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for system, row in results.iterrows():
    axes[0].scatter(row['ms_per_frame'], row['S'], s=90)
    axes[0].annotate(system, (row['ms_per_frame'], row['S']), xytext=(5, 5), textcoords='offset points')
    axes[1].scatter(row['peak_memory_mb'], row['S'], s=90)
    axes[1].annotate(system, (row['peak_memory_mb'], row['S']), xytext=(5, 5), textcoords='offset points')
axes[0].set(title='Quality vs inference latency', xlabel='Milliseconds per output frame', ylabel='Video-weighted S')
axes[1].set(title='Quality vs evaluation memory', xlabel='Peak GPU memory (MiB)', ylabel='Video-weighted S')
for ax in axes: ax.grid(alpha=0.25)
plt.tight_layout()

In [ ]:
# Smoothed training trajectories from the same four run directories.
fig, ax = plt.subplots(figsize=(10, 5))
for system in system_order:
    run_dir = summaries[system][0].parent
    log = pd.read_json(run_dir / 'train_log.jsonl', lines=True)
    ax.plot(log['global_step'], log['mean_session_loss'], label=system, linewidth=2)
ax.set(title='Exploratory training trajectories', xlabel='Training step', ylabel='Mean session loss')
ax.legend(frameon=False, ncol=4)
ax.grid(alpha=0.25)
plt.tight_layout()